# Python National Bank – Complete Banking Project

**Student name:** Abuhurairah Rahman Pathan
**Platform:** Google Colab (Python 3)

A menu-driven banking simulation. You can open accounts, deposit, withdraw, transfer with fees, check balances and tiers, print mini statements, pay savings interest and run a bank report. Everything is stored in dictionaries, and custom exceptions stop bad input from crashing the program.

**How to run:** Runtime → Run all. The last cell starts the bank menu.

## 1. Bank details and constants
Fixed settings the whole program uses.
- **Tuples** (`ACCOUNT_TYPES`, `BRANCHES`) hold values that never change.
- **Sets** (`VALID_TYPES`, `BLACKLIST`) give fast "is this allowed?" checks.
- `MIN_BALANCE` is a dictionary: Savings must keep 1,000, Current 5,000, Salary 0.
- `accounts` is the main database. Key = account number, value = that customer's details.
- `used_numbers` is a set, so no account number can ever repeat.

In [ ]:
# ============================================================
#                 PYTHON NATIONAL BANK
#              COMPLETE BANKING PROJECT
# ============================================================

from itertools import islice


# ============================================================
# 1. BANK DETAILS AND CONSTANTS
# ============================================================

BANK_NAME = "Python National Bank"
IFSC_PREFIX = "PYNB"

branch_code = 101
interest_rate = 4.0
is_open = True

# Tuples - fixed values
ACCOUNT_TYPES = ("Savings", "Current", "Salary")
BRANCHES = ("London", "Manchester", "Edinburgh")

# Sets - unique values
VALID_TYPES = set(ACCOUNT_TYPES)
BLACKLIST = {"admin", "test", "fraud"}

# Minimum balance rules
MIN_BALANCE = {
    "Savings": 1000,
    "Current": 5000,
    "Salary": 0
}

# Maximum withdrawal amount
WITHDRAW_LIMIT = 50000

# Main banking database
accounts = {}

# Store unique account numbers
used_numbers = set()

## 2. Basic helper functions
Small reusable tools.
- `banner()` prints a centred heading.
- `format_money()` shows `1500` as `$1,500.00`.
- `clean_name()` turns `'  rohan sharma '` into `'Rohan Sharma'`.
- `validate_pin()` returns True only for exactly 4 digits.
- `check_tier()` uses `if/elif/else` to label customers: Platinum (100k+), Gold (50k+), Silver (10k+), Basic.

In [ ]:
# ============================================================
# 2. BASIC FUNCTIONS
# ============================================================

def banner(title, width=50):
    """Display a formatted heading."""
    print("\n" + "=" * width)
    print(title.center(width))
    print("=" * width)


def format_money(amount):
    """Format an amount as money."""
    return f"${amount:,.2f}"


def clean_name(name):
    """Remove extra spaces and convert name to title case."""
    return name.strip().title()


def validate_pin(pin):
    """PIN must contain exactly four digits."""
    pin = str(pin).strip()
    return pin.isdigit() and len(pin) == 4


def check_tier(balance):
    """Return customer tier according to account balance."""

    if balance >= 100000:
        return "Platinum"

    elif balance >= 50000:
        return "Gold"

    elif balance >= 10000:
        return "Silver"

    else:
        return "Basic"

## 3. Unique account numbers
Builds numbers like `PYNB0001`. `{serial:04d}` pads with zeros to 4 digits. The `while` loop keeps adding 1 until it finds a number not already in `used_numbers`, then stores it.

In [ ]:
# ============================================================
# 3. GENERATE UNIQUE ACCOUNT NUMBER
# ============================================================

def generate_account_number(used_numbers):

    serial = len(used_numbers) + 1

    account_number = f"{IFSC_PREFIX}{serial:04d}"

    while account_number in used_numbers:

        serial += 1

        account_number = f"{IFSC_PREFIX}{serial:04d}"

    used_numbers.add(account_number)

    return account_number

## 4. `*args` and lambda functions
- `total_of(*amounts)` accepts any number of values and adds them (used for total interest paid).
- A **lambda** is a one-line function.
  - `simple_interest` = principal × rate × years ÷ 100.
  - `transfer_fee` = free up to 10,000, otherwise 0.5% of the amount.
  - `is_rich` checks for 50,000 or more.

In [ ]:
# ============================================================
# 4. *ARGS AND LAMBDA FUNCTIONS
# ============================================================

def total_of(*amounts):
    """Add any number of amounts."""
    return sum(amounts)


# Simple Interest
simple_interest = lambda principal, rate, years: (
    principal * rate * years / 100
)


# Transfer fee
transfer_fee = lambda amount: (
    0
    if amount <= 10000
    else round(amount * 0.005, 2)
)


# Rich customer check
is_rich = lambda balance: balance >= 50000

## 5. Custom exceptions
Our own error types, all built on `BankError`. Because they share one parent, `main()` can catch every bank error with a single `except BankError`.

Real-world analogy: one complaints desk (`BankError`) handles four kinds of complaint.

In [ ]:
# ============================================================
# 5. CUSTOM EXCEPTIONS
# ============================================================

class BankError(Exception):
    """Base class for banking errors."""
    pass


class InsufficientFundsError(BankError):
    pass


class InvalidAmountError(BankError):
    pass


class AccountNotFoundError(BankError):
    pass


class AuthenticationError(BankError):
    pass

## 6. Validation
Gatekeepers that run before any money moves.
- `get_account()` accepts lower case (`pynb0001` works) and raises `AccountNotFoundError` if the number doesn't exist.
- `validate_amount()` uses `try-except` to reject text and rejects zero or negatives.
- `verify_pin()` raises `AuthenticationError` if the PIN is wrong.

In [ ]:
# ============================================================
# 6. ACCOUNT AND INPUT VALIDATION
# ============================================================

def get_account(acc_no):
    """Find and return an account."""

    acc_no = str(acc_no).strip().upper()

    if acc_no not in accounts:

        raise AccountNotFoundError(
            f"Account {acc_no} not found."
        )

    return accounts[acc_no]


def validate_amount(value):
    """Check that amount is numeric and greater than zero."""

    try:

        amount = float(value)

    except (ValueError, TypeError):

        raise InvalidAmountError(
            "Amount must be a number."
        )

    if amount <= 0:

        raise InvalidAmountError(
            "Amount must be greater than zero."
        )

    return round(amount, 2)


def verify_pin(account, pin):
    """Verify customer PIN."""

    if account["pin"] != str(pin).strip():

        raise AuthenticationError(
            "Incorrect PIN."
        )

## 7. Create account
Checks run in order, and any failure stops the process with an error:
1. The name is cleaned, must be 3+ letters, and must not contain a blacklisted word (set intersection `&`).
2. The type must be Savings, Current or Salary.
3. The PIN must be 4 digits.
4. The opening deposit must meet that type's minimum balance.

Only then is an account number generated and the record saved, with an `Opening` entry in its history.

In [ ]:
# ============================================================
# 7. CREATE ACCOUNT
# ============================================================

def create_account(name, acc_type, opening, pin):

    # Clean customer name
    name = clean_name(name)

    # Validate name
    if len(name) < 3 or not name.replace(" ", "").isalpha():

        raise ValueError(
            "Name must contain only letters and spaces "
            "(minimum 3 characters)."
        )

    # Check blacklist
    if set(name.lower().split()) & BLACKLIST:

        raise ValueError(
            "This name is not allowed."
        )

    # Format account type
    acc_type = acc_type.strip().title()

    # Validate account type
    if acc_type not in VALID_TYPES:

        raise ValueError(
            "Account type must be one of: "
            + ", ".join(ACCOUNT_TYPES)
        )

    # Validate PIN
    pin = str(pin).strip()

    if not validate_pin(pin):

        raise ValueError(
            "PIN must be exactly 4 digits."
        )

    # Validate opening balance
    opening = validate_amount(opening)

    # Check minimum balance
    if opening < MIN_BALANCE[acc_type]:

        raise InvalidAmountError(
            f"{acc_type} account requires a minimum balance of "
            f"{format_money(MIN_BALANCE[acc_type])}."
        )

    # Generate account number
    acc_no = generate_account_number(
        used_numbers
    )

    # Store account information
    accounts[acc_no] = {

        "name": name,

        "type": acc_type,

        "pin": pin,

        "balance": opening,

        "history": [
            (
                "Opening",
                opening,
                opening
            )
        ]
    }

    return acc_no

## 8. Deposit
Find the account → validate the amount → add it → record `(type, amount, new balance)` as a tuple in `history`. No PIN is needed, just like paying cash in at a branch.

In [ ]:
# ============================================================
# 8. DEPOSIT
# ============================================================

def deposit(acc_no, amount):

    account = get_account(acc_no)

    amount = validate_amount(amount)

    # Add amount
    account["balance"] += amount

    # Add transaction to history
    account["history"].append(
        (
            "Deposit",
            amount,
            account["balance"]
        )
    )

    return account["balance"]

## 9. Withdraw
The PIN is checked first. The amount must be under the 50,000 daily limit, and the account must keep its minimum balance afterwards. Only then is the money deducted and logged.

In [ ]:
# ============================================================
# 9. WITHDRAW
# ============================================================

def withdraw(acc_no, pin, amount):

    account = get_account(acc_no)

    # Check PIN
    verify_pin(account, pin)

    amount = validate_amount(amount)

    # Check withdrawal limit
    if amount > WITHDRAW_LIMIT:

        raise InvalidAmountError(
            f"Daily withdrawal limit is "
            f"{format_money(WITHDRAW_LIMIT)}."
        )

    # Check minimum account balance
    if (
        account["balance"] - amount
        < MIN_BALANCE[account["type"]]
    ):

        raise InsufficientFundsError(
            f"Insufficient funds. "
            f"{account['type']} accounts must maintain "
            f"{format_money(MIN_BALANCE[account['type']])}."
        )

    # Deduct amount
    account["balance"] -= amount

    # Save transaction
    account["history"].append(
        (
            "Withdraw",
            amount,
            account["balance"]
        )
    )

    return account["balance"]

## 10. Transfer
- Both accounts must exist and must be different.
- The sender's PIN is checked.
- The fee is added on top: sending 20,000 costs 20,100.
- The sender is charged amount + fee, the receiver gets the amount, and both histories are updated.
- The function returns the fee so the menu can display it.

In [ ]:
# ============================================================
# 10. TRANSFER MONEY
# ============================================================

def transfer(from_no, pin, to_no, amount):

    source_account = get_account(from_no)

    destination_account = get_account(to_no)

    # Prevent transfer to same account
    if source_account is destination_account:

        raise InvalidAmountError(
            "Cannot transfer money to the same account."
        )

    # Verify source PIN
    verify_pin(source_account, pin)

    amount = validate_amount(amount)

    # Calculate transfer fee
    fee = transfer_fee(amount)

    total_amount = amount + fee

    # Check available balance
    if (
        source_account["balance"] - total_amount
        < MIN_BALANCE[source_account["type"]]
    ):

        raise InsufficientFundsError(
            "Insufficient funds for transfer "
            "(amount + transfer fee)."
        )

    # Deduct money from sender
    source_account["balance"] -= total_amount

    source_account["history"].append(
        (
            "Transfer Out",
            total_amount,
            source_account["balance"]
        )
    )

    # Add money to receiver
    destination_account["balance"] += amount

    destination_account["history"].append(
        (
            "Transfer In",
            amount,
            destination_account["balance"]
        )
    )

    return fee

## 11. Savings interest
Loops through every account, and only `Savings` accounts receive simple interest (4% for 1 year by default). Returns a dictionary `{account number: interest paid}`.

**Note:** each time you choose option 7 it pays interest again. This is a simulation, so there's no once-a-year rule.

In [ ]:
# ============================================================
# 11. APPLY SAVINGS INTEREST
# ============================================================

def apply_interest(rate=4.0, years=1):

    credited = {}

    for acc_no, account in accounts.items():

        if account["type"] == "Savings":

            interest = round(
                simple_interest(
                    account["balance"],
                    rate,
                    years
                ),
                2
            )

            account["balance"] += interest

            account["history"].append(
                (
                    "Interest",
                    interest,
                    account["balance"]
                )
            )

            credited[acc_no] = interest

    return credited

## 12. Statement
`last_n=None` shows the full history. `last_n=5` uses `reversed()` + `islice()` to take the newest 5 transactions, then `[::-1]` puts them back in date order. f-string widths (`:<16`, `:>14,.2f`) line the columns up.

In [ ]:
# ============================================================
# 12. PRINT BANK STATEMENT
# ============================================================

def print_statement(acc_no, last_n=None):

    acc_no = str(acc_no).strip().upper()

    account = get_account(acc_no)

    # All transactions or only last N transactions
    if last_n is None:

        rows = account["history"]

    else:

        rows = list(
            islice(
                reversed(account["history"]),
                last_n
            )
        )[::-1]

    banner(
        f"STATEMENT - {acc_no}"
    )

    print(
        f"Name: {account['name']} | "
        f"Type: {account['type']}"
    )

    print(
        f"{'#':<4}"
        f"{'Type':<16}"
        f"{'Amount':>14}"
        f"{'Balance':>16}"
    )

    print("-" * 52)

    for number, (
        transaction_type,
        amount,
        balance
    ) in enumerate(
        rows,
        start=1
    ):

        print(
            f"{number:<4}"
            f"{transaction_type:<16}"
            f"{amount:>14,.2f}"
            f"{balance:>16,.2f}"
        )

    print("-" * 52)

    print(
        "Closing balance:",
        format_money(
            account["balance"]
        )
    )

## 13. Bank report
Everything is calculated live:
- Total deposits use a **generator** inside `sum()`.
- Accounts by type use a **dictionary comprehension**.
- The richest customer uses `max()` with a lambda key.
- The ranking uses `sorted(..., reverse=True)`.

In [ ]:
# ============================================================
# 13. BANK REPORT
# ============================================================

def bank_report():

    banner(
        "BANK REPORT"
    )

    if not accounts:

        print(
            "No customer accounts available."
        )

        return

    # Total deposits using generator
    total = sum(
        account["balance"]
        for account in accounts.values()
    )

    # Count accounts by type using dictionary comprehension
    count_by_type = {

        account_type: sum(
            1
            for account in accounts.values()
            if account["type"] == account_type
        )

        for account_type in ACCOUNT_TYPES
    }

    # Richest customer using max() and lambda
    richest_no, richest = max(

        accounts.items(),

        key=lambda item:
        item[1]["balance"]
    )

    print(
        f"Total customers : "
        f"{len(accounts)}"
    )

    print(
        f"Total deposits  : "
        f"{format_money(total)}"
    )

    print(
        f"Accounts by type: "
        f"{count_by_type}"
    )

    print(
        f"Richest customer: "
        f"{richest['name']} "
        f"({format_money(richest['balance'])})"
    )

    print()

    print(
        f"{'Rank':<6}"
        f"{'Acc No':<12}"
        f"{'Name':<20}"
        f"{'Tier':<12}"
        f"{'Balance':>14}"
    )

    print("-" * 64)

    # Sort accounts from highest to lowest balance
    ranking = sorted(

        accounts.items(),

        key=lambda item:
        item[1]["balance"],

        reverse=True
    )

    for rank, (
        acc_no,
        account
    ) in enumerate(
        ranking,
        start=1
    ):

        print(
            f"{rank:<6}"
            f"{acc_no:<12}"
            f"{account['name']:<20}"
            f"{check_tier(account['balance']):<12}"
            f"{account['balance']:>14,.2f}"
        )

## 14. Demo customers
Five ready-made customers so you can test straight away.

| Account | Name | Type | Opening | PIN |
|---|---|---|---|---|
| PYNB0001 | Rohan Sharma | Savings | 75,000 | 1234 |
| PYNB0002 | Simran Kaur | Current | 120,000 | 2222 |
| PYNB0003 | Aman Verma | Salary | 15,000 | 3333 |
| PYNB0004 | Neha Gupta | Savings | 5,000 | 4444 |
| PYNB0005 | Karan Singh | Current | 48,000 | 5555 |

In [ ]:
# ============================================================
# 14. DEMONSTRATION CUSTOMERS
# ============================================================

def load_demo_customers():

    demo_customers = [

        (
            "Rohan Sharma",
            "Savings",
            75000,
            "1234"
        ),

        (
            "Simran Kaur",
            "Current",
            120000,
            "2222"
        ),

        (
            "Aman Verma",
            "Salary",
            15000,
            "3333"
        ),

        (
            "Neha Gupta",
            "Savings",
            5000,
            "4444"
        ),

        (
            "Karan Singh",
            "Current",
            48000,
            "5555"
        )
    ]

    for (
        name,
        account_type,
        balance,
        pin
    ) in demo_customers:

        create_account(
            name,
            account_type,
            balance,
            pin
        )

## 15. Menu
A list of option names printed with `enumerate(start=1)`, so the numbers are generated automatically, plus `0. Exit`.

In [ ]:
# ============================================================
# 15. DISPLAY MENU
# ============================================================

def print_menu():

    banner(
        "🏦 " + BANK_NAME
    )

    menu = [

        "Create Account",

        "Deposit",

        "Withdraw",

        "Transfer",

        "Check Balance",

        "Mini Statement",

        "Apply Savings Interest",

        "Bank Report"
    ]

    for number, option in enumerate(
        menu,
        start=1
    ):

        print(
            f"  {number}. {option}"
        )

    print(
        "  0. Exit"
    )

## 16. Main program
A `while True` loop shows the menu and runs the chosen option. All the work sits inside one `try`:
- `except BankError` catches money, PIN and account errors and prints the error type and message.
- `except ValueError` catches name, type and PIN-format errors.

So invalid input never crashes the program. Option `0` uses `break` to exit.

In [ ]:
# ============================================================
# 16. MAIN BANKING PROGRAM
# ============================================================

def main():

    while True:

        print_menu()

        choice = input(
            "\nEnter choice: "
        ).strip()

        try:

            # =================================================
            # OPTION 1 - CREATE ACCOUNT
            # =================================================

            if choice == "1":

                name = input(
                    "Customer name   : "
                )

                acc_type = input(
                    f"Type {ACCOUNT_TYPES}: "
                )

                opening = input(
                    "Opening deposit : "
                )

                pin = input(
                    "Set 4-digit PIN : "
                )

                acc_no = create_account(
                    name,
                    acc_type,
                    opening,
                    pin
                )

                print(
                    f"✅ Account created successfully!"
                )

                print(
                    f"Your account number is {acc_no}"
                )


            # =================================================
            # OPTION 2 - DEPOSIT
            # =================================================

            elif choice == "2":

                acc_no = input(
                    "Account number : "
                )

                amount = input(
                    "Amount         : "
                )

                balance = deposit(
                    acc_no,
                    amount
                )

                print(
                    "✅ Deposit successful."
                )

                print(
                    "Balance:",
                    format_money(balance)
                )


            # =================================================
            # OPTION 3 - WITHDRAW
            # =================================================

            elif choice == "3":

                acc_no = input(
                    "Account number : "
                )

                pin = input(
                    "PIN            : "
                )

                amount = input(
                    "Amount         : "
                )

                balance = withdraw(
                    acc_no,
                    pin,
                    amount
                )

                print(
                    "✅ Please collect your cash."
                )

                print(
                    "Balance:",
                    format_money(balance)
                )


            # =================================================
            # OPTION 4 - TRANSFER
            # =================================================

            elif choice == "4":

                source = input(
                    "From account   : "
                )

                pin = input(
                    "PIN            : "
                )

                destination = input(
                    "To account     : "
                )

                amount = input(
                    "Amount         : "
                )

                fee = transfer(
                    source,
                    pin,
                    destination,
                    amount
                )

                print(
                    "✅ Transfer successful."
                )

                print(
                    "Fee charged:",
                    format_money(fee)
                )


            # =================================================
            # OPTION 5 - CHECK BALANCE
            # =================================================

            elif choice == "5":

                account = get_account(
                    input(
                        "Account number : "
                    )
                )

                verify_pin(
                    account,
                    input(
                        "PIN            : "
                    )
                )

                print(
                    f"💰 {account['name']} | "
                    f"{check_tier(account['balance'])} tier | "
                    f"{format_money(account['balance'])}"
                )


            # =================================================
            # OPTION 6 - MINI STATEMENT
            # =================================================

            elif choice == "6":

                acc_no = input(
                    "Account number : "
                )

                account = get_account(
                    acc_no
                )

                verify_pin(
                    account,
                    input(
                        "PIN            : "
                    )
                )

                print_statement(
                    acc_no,
                    last_n=5
                )


            # =================================================
            # OPTION 7 - SAVINGS INTEREST
            # =================================================

            elif choice == "7":

                credited = apply_interest(
                    rate=interest_rate
                )

                if credited:

                    for (
                        acc_no,
                        interest
                    ) in credited.items():

                        print(
                            f"✅ {acc_no} "
                            f"({accounts[acc_no]['name']}): "
                            f"interest credited "
                            f"{format_money(interest)}"
                        )

                    print(
                        "Total interest paid:",
                        format_money(
                            total_of(*credited.values())
                        )
                    )

                else:

                    print(
                        "No Savings accounts found."
                    )


            # =================================================
            # OPTION 8 - BANK REPORT
            # =================================================

            elif choice == "8":

                bank_report()


            # =================================================
            # OPTION 0 - EXIT
            # =================================================

            elif choice == "0":

                print(
                    f"\nThank you for banking with "
                    f"{BANK_NAME}. Goodbye! 👋"
                )

                break


            # =================================================
            # INVALID MENU CHOICE
            # =================================================

            else:

                print(
                    "❌ Invalid choice. Please enter "
                    "a number from 0 to 8."
                )


        # =====================================================
        # ERROR HANDLING - program never crashes
        # =====================================================

        except BankError as error:

            # Catches all custom bank errors:
            # InsufficientFundsError, InvalidAmountError,
            # AccountNotFoundError, AuthenticationError
            print(
                f"❌ {type(error).__name__}: {error}"
            )

        except ValueError as error:

            # Catches name, account type and PIN errors
            print(
                f"❌ Input error: {error}"
            )

## 17. Start the program
Loads the demo customers (only if `accounts` is empty, so re-running this cell doesn't create duplicates), then starts `main()`.

**Run this cell last.**

In [ ]:
# ============================================================
# 17. START THE PROGRAM
# ============================================================

if __name__ == "__main__":

    # Load demo customers only once
    # (stops duplicates if this cell is run again)
    if not accounts:

        load_demo_customers()

    print(
        f"{len(accounts)} demo customers loaded: "
        f"{', '.join(accounts)}"
    )

    main()

## Test checklist
| # | Test | Input | Expected |
|---|---|---|---|
| 1 | Create account | 1 → `priya patel`, `savings`, `2000`, `7777` | PYNB0006 created, name tidied |
| 2 | Bad name / PIN / type | `Ab`, `Test User`, PIN `12a4`, type `Gold` | Input error, no account |
| 3 | Below minimum | Current with `4000` | InvalidAmountError |
| 4 | Deposit | 2 → `pynb0004` → `500` | Balance $5,500.00 |
| 5 | Text amount | 2 → `PYNB0004` → `abc` | Amount must be a number |
| 6 | Withdraw | 3 → `PYNB0001`, `1234`, `5000` | Balance $70,000.00 |
| 7 | Wrong PIN | 3 → `PYNB0001`, `0000` | AuthenticationError |
| 8 | Over limit | 3 → `PYNB0002`, `2222`, `60000` | Limit message |
| 9 | Below minimum on withdraw | 3 → `PYNB0004`, `4444`, `5000` | InsufficientFundsError |
| 10 | Transfer with fee | 4 → `PYNB0002`, `2222`, `PYNB0003`, `20000` | Fee $100.00 |
| 11 | Transfer, no fee | 4 → `PYNB0003`, `3333`, `PYNB0005`, `5000` | Fee $0.00 |
| 12 | Same account | 4 → `PYNB0001` to `PYNB0001` | Blocked |
| 13 | Check balance | 5 → `PYNB0002`, `2222` | Name, tier, balance |
| 14 | Mini statement | 6 → `PYNB0004`, `4444` | Last 5 transactions |
| 15 | Interest | 7 | Savings accounts only get 4% |
| 16 | Bank report | 8 | Totals, counts, richest, ranking |
| 17 | Invalid choice | `9` or `abc` | Choose 0 to 8 |
| 18 | Exit | 0 | Goodbye message |